# Lab 12 - Real Deployment: RAG Agent with Amazon Bedrock
**Sections covered: 11 (Real Deployments) + 10 (How to Build)**

---

## What this lab demonstrates

In Section 11 we covered three real-world agent patterns:
- **Pattern A** - Conversational agent with RAG knowledge retrieval
- **Pattern B** - Event-driven backend agent
- **Pattern C** - Natural language to data query

This lab builds **Pattern A** end-to-end using **Amazon Bedrock** -
the managed AWS service for running foundation models in production.

You will build an IT support agent that:
1. Queries a knowledge base (RAG) before answering policy questions
2. Calls Claude via **Amazon Bedrock** using IAM auth (no API keys in code)
3. Raises support tickets and checks ticket status via tools
4. Runs the same Reason -> Act -> Observe loop you built in Labs 2-10

---

## Why Bedrock?

In production you never hardcode API keys. You use **IAM roles**.
Bedrock integrates natively with S3, Lambda, OpenSearch, and CloudWatch.
Amazon Bedrock Knowledge Base gives you managed RAG infrastructure
without running your own vector database.

---

## Prerequisites
- AWS CLI configured (`aws configure` done, or IAM role attached to your environment)
- `boto3` installed: `pip install boto3`
- Bedrock model access enabled: AWS Console -> Amazon Bedrock -> Model access
  -> Request access for `anthropic.claude-sonnet-4-5` in your region
- Recommended region: `us-east-1`

## Step 1 - Setup and Bedrock Connection

In [ ]:
import boto3
import json
from datetime import datetime

# AWS Configuration
REGION = "us-east-1"          # Change to your region
MODEL  = "anthropic.claude-sonnet-4-5"

# boto3 uses your AWS CLI credentials / IAM role automatically.
# No API keys in code - this is the production pattern.
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

# Quick connectivity test
try:
    test_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 32,
        "messages": [{"role": "user", "content": "Reply with: OK"}]
    }
    resp = bedrock.invoke_model(modelId=MODEL, body=json.dumps(test_body))
    result = json.loads(resp["body"].read())
    print(f"Bedrock connection: {result['content'][0]['text']}")
    print(f"Model: {MODEL}  |  Region: {REGION}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Check: AWS CLI configured? Bedrock model access enabled?")

## Step 2 - The Knowledge Base (RAG Layer)

In production this is an **Amazon Bedrock Knowledge Base**:
S3 bucket (documents) -> Titan Embeddings (vectors) -> OpenSearch Serverless (search).

Here we simulate the retrieval function so you can run this without provisioning infrastructure.
The concept and the function signature are identical to the real Bedrock KB API.

In [ ]:
# Simulated knowledge base
# In production: Amazon Bedrock Knowledge Base backed by S3 + OpenSearch Serverless + Titan embeddings
# The retrieval function below simulates the KB.retrieve() API call

KNOWLEDGE_BASE = {
    "laptop_policy": {
        "title": "Laptop Replacement Policy",
        "content": (
            "Employees are eligible for a laptop replacement every 3 years or if the device "
            "is confirmed faulty by IT. Replacement requests go through the IT portal. "
            "Standard processing: 5 business days. Emergency replacements for critical roles: "
            "24 hours with line manager approval. Budget limit: 1200 GBP per device."
        )
    },
    "software_request": {
        "title": "Software Request Process",
        "content": (
            "Software requests need line manager approval and an IT security review. "
            "Standard approved software is available immediately via self-service. "
            "New software takes 3-5 days for security review. "
            "Software over 500 GBP/year also needs Finance approval. "
            "Open-source tools need a licence compatibility check."
        )
    },
    "remote_access": {
        "title": "Remote Access and VPN",
        "content": (
            "VPN access uses the company-issued Cisco AnyConnect client. "
            "First-time setup: IT must whitelist your device. "
            "Contact helpdesk with your device serial number and employee ID. "
            "MFA is mandatory. If locked out, call the 24/7 helpdesk: ext 4444."
        )
    },
    "password_reset": {
        "title": "Password Reset Procedure",
        "content": (
            "Self-service reset is at reset.company.com using your mobile or backup email. "
            "Account locks after 5 failed attempts and resets after 30 minutes. "
            "For immediate unlock, call ext 4444. "
            "Passwords must be 12+ characters with uppercase, lowercase, number, and symbol."
        )
    }
}

def rag_retrieve(query: str) -> str:
    """
    Simulates vector similarity search against a knowledge base.
    Production equivalent: boto3 bedrock-agent-runtime.retrieve()
    with a Bedrock Knowledge Base ID.
    """
    query_lower = query.lower()
    results = []
    for key, doc in KNOWLEDGE_BASE.items():
        score = sum(1 for word in query_lower.split()
                    if word in doc["content"].lower() or word in doc["title"].lower())
        if score > 0:
            results.append((score, doc))
    if not results:
        return "No relevant policy found in the knowledge base for this query."
    results.sort(key=lambda x: x[0], reverse=True)
    top = results[0][1]
    return f"[KB: {top['title']}]\n{top['content']}"

print("Knowledge base loaded:")
for doc in KNOWLEDGE_BASE.values():
    print(f"  - {doc['title']}")
print("\nIn production: replace rag_retrieve() with Bedrock Knowledge Base retrieve() API")

## Step 3 - Tools

Three tools: knowledge base lookup, raise a ticket, check a ticket.

In [ ]:
TOOLS = [
    {
        "name": "rag_lookup",
        "description": (
            "Search the company knowledge base for policies and procedures. "
            "Always use this before answering any policy question."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What to search for"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "raise_ticket",
        "description": "Create a support ticket for issues needing IT action.",
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "enum": ["hardware", "software", "access", "account", "other"]
                },
                "summary": {"type": "string", "description": "One-line issue summary"},
                "priority": {
                    "type": "string",
                    "enum": ["low", "normal", "high", "critical"]
                }
            },
            "required": ["category", "summary", "priority"]
        }
    },
    {
        "name": "check_ticket",
        "description": "Check status of an existing support ticket by ticket ID.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string", "description": "Ticket ID e.g. TKT-1042"}
            },
            "required": ["ticket_id"]
        }
    }
]

TICKETS = {
    "TKT-1042": {"status": "In Progress", "assigned_to": "Sarah (IT)", "eta": "2 business days"},
    "TKT-0891": {"status": "Resolved", "resolution": "VPN client reinstalled"},
    "TKT-1105": {"status": "Waiting for approval", "approver": "Line Manager"},
}
ticket_counter = [1200]

def execute_tool(name: str, inputs: dict) -> str:
    if name == "rag_lookup":
        result = rag_retrieve(inputs["query"])
        print(f"  [RAG] {result[:80]}...")
        return result
    elif name == "raise_ticket":
        ticket_counter[0] += 1
        tid = f"TKT-{ticket_counter[0]}"
        eta = "2 hours" if inputs["priority"] == "critical" else "1 business day"
        print(f"  [TICKET] Created {tid}: {inputs['summary']} [{inputs['priority']}]")
        return f"Ticket {tid} created. Category: {inputs['category']}. Priority: {inputs['priority']}. ETA: {eta}"
    elif name == "check_ticket":
        tid = inputs["ticket_id"].upper()
        ticket = TICKETS.get(tid)
        if not ticket:
            return f"Ticket {tid} not found."
        print(f"  [TICKET] {tid}: {ticket['status']}")
        return f"Ticket {tid}: {json.dumps(ticket)}"
    return f"Unknown tool: {name}"

print("Tools ready")

## Step 4 - The Bedrock Agent Loop

Same loop as Labs 2-10. The only difference: `call_bedrock()` instead of `client.messages.create()`.
The model, tools, messages, and stop reasons are identical in structure.

In [ ]:
def call_bedrock(messages, tools, system):
    """Wraps bedrock.invoke_model in the same interface as the Anthropic SDK."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1024,
        "system": system,
        "tools": tools,
        "messages": messages
    }
    response = bedrock.invoke_model(modelId=MODEL, body=json.dumps(body))
    return json.loads(response["body"].read())


def run_support_agent(user_message: str, max_turns: int = 10) -> str:
    """
    IT support agent on Amazon Bedrock.
    Same Reason -> Act -> Observe loop as all previous labs.
    The only difference: call_bedrock() instead of client.messages.create()
    """
    SYSTEM = (
        "You are an IT support assistant. "
        "Always use rag_lookup first before answering policy questions - never answer from memory. "
        "Raise a ticket when the user has an issue needing action, not just information. "
        "Be concise and specific."
    )

    messages = [{"role": "user", "content": user_message}]
    print(f"\nUser: {user_message}")

    for turn in range(max_turns):
        response = call_bedrock(messages, TOOLS, SYSTEM)
        stop_reason = response.get("stop_reason")
        content = response.get("content", [])
        messages.append({"role": "assistant", "content": content})

        if stop_reason == "end_turn":
            for block in content:
                if block.get("type") == "text":
                    print(f"\nAgent: {block['text']}")
                    return block["text"]
            return ""

        if stop_reason == "tool_use":
            tool_results = []
            for block in content:
                if block.get("type") == "tool_use":
                    result = execute_tool(block["name"], block["input"])
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block["id"],
                        "content": result
                    })
            messages.append({"role": "user", "content": tool_results})

    return "Max turns reached"

print("Bedrock agent ready - run the cells below")

## Step 5 - Run Three Scenarios

In [ ]:
# Scenario A - Policy question (agent uses RAG before answering)
run_support_agent("My laptop is 2 years old and running very slowly. Can I get a new one?")

In [ ]:
# Scenario B - Urgent issue requiring a ticket
run_support_agent(
    "I have been locked out of my account for 3 hours. "
    "Self-service reset is not working. I have a client call in 45 minutes."
)

In [ ]:
# Scenario C - Check existing ticket + follow-up policy question in one conversation
run_support_agent(
    "Can you check ticket TKT-1042 for me? "
    "Also, what is the process if I need new software for a project?"
)

## Step 6 - Bedrock vs Direct SDK: When to Use Each

In [ ]:
print("""
Anthropic SDK (direct API key)          Amazon Bedrock (boto3 + IAM)
================================        ================================
+ Simple setup for learning             + IAM roles - no keys in code
+ API key in .env                       + Native AWS service integration
+ Latest features immediately           + Bedrock Knowledge Base (managed RAG)
- Key management risk in prod           + CloudWatch logging built-in
- No AWS native integration             + VPC, compliance certifications
                                        + One AWS bill
                                        - Model access setup needed

When to use Bedrock:
  Building on AWS, need IAM auth, enterprise compliance,
  want managed RAG (Bedrock Knowledge Base), Bedrock Agents runtime.

What Bedrock Agents gives you on top of this lab:
  - The agent loop managed for you (no loop code)
  - Lambda functions as tools (serverless, auto-scaling)
  - Knowledge Base connected natively (S3 + OpenSearch)
  - CloudWatch traces for every agent step
  - Guardrails (content filtering, PII redaction)

This lab showed you what is INSIDE Bedrock Agents.
""")

## Key Takeaways

| What you built | Production equivalent |
|----------------|----------------------|
| `rag_retrieve()` | Bedrock Knowledge Base `retrieve()` API |
| `raise_ticket()` | AWS Lambda function triggered by agent |
| `call_bedrock()` | `bedrock-runtime.invoke_model()` with IAM auth |
| The agent loop | What Amazon Bedrock Agents manages for you |
| `SYSTEM` prompt | Bedrock Agents "agent instruction" |

---
**Next:** Lab 13 - Risks: Prompt Injection and Guardrails
You will see a real prompt injection attack succeed, then build defences against it.